In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [24]:

data = pd.read_csv("data.csv")
data = data.drop(columns=['paper', 'catalyst used'])

In [25]:
data['temperature(process)'] = pd.to_numeric(data['temperature(process)'], errors='coerce')
data['pressure(process)'] = pd.to_numeric(data['pressure(process)'], errors='coerce')

# Clean "conversion rate1" to extract numeric values
def clean_conversion_rate(value):
    if isinstance(value, str):
        value = value.replace('%', '').strip()
        if "to" in value:
            return np.mean([float(v) for v in value.split('to')])
        try:
            return float(value)
        except ValueError:
            return np.nan
    return value

In [26]:
data['conversion_rate'] = data['conversion rate1'].apply(clean_conversion_rate)

# Drop the old "conversion rate1" column
data = data.drop(columns=['conversion rate1'])

# Rename columns for consistency
data.columns = data.columns.str.strip().str.replace(' ', '_').str.replace('(process)', '').str.lower()

# Drop rows with missing values
data = data.dropna()

In [27]:
# Split the dataset into features (X) and target (y)
X = data.drop(columns=['conversion_rate'])
y = data['conversion_rate']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [28]:
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

In [29]:
# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error (MSE):", mse)
print("R-squared (R2):", r2)

Mean Squared Error (MSE): 1585.9695886499999
R-squared (R2): 0.017954204177799693


In [41]:
# Predict conversion rate for new input
temperature = 250  # Example: Temperature in °C
pressure = 2   # Example: Pressure in MPa
surface_area = 2382  # Example: Surface area in m²/g
reaction_time = 2   # Example: Reaction time in hours
lignin_loading = 0.5  # Example: Lignin loading in wt%
catalyst = "ZrO2, WOx, and MoO3"

# Prepare input as a DataFrame with the same column names used during training
input_data = pd.DataFrame([[temperature, pressure, surface_area, reaction_time, lignin_loading]],
                          columns=X_train.columns)

# Predict the conversion rate
predicted_conversion_rate = model.predict(input_data)
print("\nPredicted Conversion Rate:", predicted_conversion_rate[0], "\nCatalyst Used:", catalyst)



Predicted Conversion Rate: 38.585199999999986 
Catalyst Used: ZrO2, WOx, and MoO3
